In [0]:
%run ../common/_utils_config

In [0]:
%run ../common/_utils_reference_data

In [0]:
%run ../bronze/utils/_utils_bronze_framework

In [0]:
dbutils.widgets.text("sources", "", "Sources (comma-separated, empty = all)")
dbutils.widgets.dropdown("mode", "auto", ["scheduled", "auto", "reconcile", "initial", "refresh"], "Mode")

In [0]:
# Bronze ingestion: every source in source_specs/ (or the selected ones), isolated from each other.
#   scheduled -> what the daily job uses: reconcile on Sundays, auto on the other days
#   auto      -> first load for new sources, refresh_window for the rest
#   reconcile -> reconcile_window (late corrections and deletions further back)
#   initial   -> manual full history reload
# Status per source: ok · deferred (API budget or block: expected, retried by later runs) ·
#                    partial (some chunks failed) · failed (the source could not run)
reconcile_weekday = 6          # Monday = 0 ... Sunday = 6

requested = [s.strip() for s in dbutils.widgets.get("sources").split(",") if s.strip()]
mode_param = dbutils.widgets.get("mode")
if mode_param == "scheduled":
    mode = "reconcile" if date.today().weekday() == reconcile_weekday else "auto"
else:
    mode = mode_param
available = list_source_specs()

unknown = [s for s in requested if s not in available]
if unknown:
    raise ValueError(f"Unknown sources {unknown}; available: {available}")
sources = requested or available
print(f"Bronze ingestion · mode {mode}" + (f" (scheduled, {date.today():%A})" if mode_param == "scheduled" else "")
      + f" · {len(sources)} source(s): {sources}\n")


def log_source_failure(source_name, error_message):
    """Failures that happen before any chunk is logged (invalid spec, framework error)."""
    rows = [(str(uuid.uuid4()), source_name, None, None, None, "failed", None, 0, None, error_message[:1000])]
    (spark.createDataFrame(rows, ingestion_log_schema)
          .withColumn("ingested_at", F.current_timestamp())
          .write.mode("append").saveAsTable(ingestion_log_table))


run_summary = []
for source_name in sources:
    started = time.time()
    try:
        result = run_source_spec(load_source_spec_file(source_name), mode=mode)
        if result["chunks_error"]:
            status = "partial"
        elif result["chunks_deferred"]:
            status = "deferred"
        else:
            status = "ok"
        run_summary.append((source_name, status, result["mode"], result["window"], result["unit"],
                            result["chunks_ok"], result["chunks_error"], result["chunks_deferred"],
                            round(time.time() - started, 1), None))
    except Exception as e:
        error_message = f"{type(e).__name__}: {e}"
        print(f"❌ {source_name}: {error_message[:300]}")
        log_source_failure(source_name, error_message)
        run_summary.append((source_name, "failed", None, None, None, 0, 0, 0, round(time.time() - started, 1),
                            error_message[:300]))
    print()

In [0]:
summary_df = spark.createDataFrame(
    run_summary,
    "source_name STRING, status STRING, run_mode STRING, window_units INT, unit STRING, "
    "chunks_ok INT, chunks_error INT, chunks_deferred INT, seconds DOUBLE, error STRING",
)
display(summary_df)

deferred = [r[0] for r in run_summary if r[1] == "deferred"]
if deferred:
    print(f"⏳ Deferred by the API budget or a block (retried automatically by later runs): {deferred}")

problems = [r for r in run_summary if r[1] in ("partial", "failed")]
if problems:
    # Failing the notebook makes the Job fail -> Databricks sends the failure notification.
    # Failed chunks are retried automatically on the next run.
    raise RuntimeError(f"Bronze ingestion finished with problems in {len(problems)} source(s): "
                       f"{[(r[0], r[1]) for r in problems]}. See {ingestion_log_table}.")

print(f"✅ Bronze ingestion OK · {len(run_summary)} source(s)")